In [1]:
def parse_aum(aum_str: str) -> float:
    """
    將帶有 M, B, K 後綴的 AUM 字串轉換為實際數值，
    若遇到 N/A 或無法解析的格式則回傳 0.0。
    """
    # 處理空值或無效值
    if not aum_str or aum_str in ["N/A", "-", "n/a"]:
        return 0.0
        
    # 移除千分位逗號並去除頭尾空白
    aum_str = aum_str.replace(",", "").strip()
    
    multiplier = 1.0
    # 判斷並處理單位後綴
    if aum_str.endswith("B"):
        multiplier = 1_000_000_000.0
        aum_str = aum_str[:-1]
    elif aum_str.endswith("M"):
        multiplier = 1_000_000.0
        aum_str = aum_str[:-1]
    elif aum_str.endswith("K"):
        multiplier = 1_000.0
        aum_str = aum_str[:-1]
        
    try:
        # 轉換為浮點數並乘上對應的倍數
        return float(aum_str) * multiplier
    except ValueError:
        return 0.0

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

def get_all_etfs() -> list[str]:
    """
    抓取所有 ETF 的資訊，並儲存完整的資料。
    """
    csv_file = "csv\\all_etfs.csv"
    if os.path.exists(csv_file):
        print(f"已找到 {csv_file}，直接讀取資料。")
        df = pd.read_csv(csv_file)
        return df["ticker"].tolist()
    base_url = "https://stockanalysis.com/etf/"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    all_results = []
    page = 1

    while True:
        print(f"正在請求第 {page} 頁...")
        resp = requests.get(f"{base_url}?page={page}", headers=headers, timeout=15)
        resp.raise_for_status()

        soup = BeautifulSoup(resp.text, "html.parser")
        table = soup.find("table")

        if not table:
            print("未找到任何資料，停止抓取。")
            break

        # 解析資料行
        rows = table.find("tbody").find_all("tr")
        if not rows:
            print("已無更多資料可抓取，停止。")
            break

        for row in rows:
            cols = row.find_all("td")
            if len(cols) < 2:
                continue

            ticker = cols[0].get_text(strip=True)
            name = cols[1].get_text(strip=True)
            asset_class = cols[2].get_text(strip=True)
            aum = cols[3].get_text(strip=True) if len(cols) > 3 else "N/A"  # 假設 AUM 在第 4 欄

            all_results.append({
                "ticker": ticker,
                "name": name,
                "asset_class": asset_class,
                "assets_aum": aum,
            })

        print(f"第 {page} 頁資料抓取成功，共計資料行數：{len(rows)}")
        if len(rows) < 500:  # 假設每頁最多 500 筆資料，少於 500 筆表示最後一頁
            break
        page += 1

    # 儲存結果
    all_results.sort(key=lambda x:parse_aum(x["assets_aum"]), reverse=True)
    df = pd.DataFrame(all_results)
    df.to_csv("csv\\all_etfs.csv", index=False)
    print("所有 ETF 資料已儲存至 all_etfs.csv")
    
    return df["ticker"].tolist()

if __name__ == "__main__":
    tickers = get_all_etfs()
    print("\n--- 代碼清單 ---")
    print(tickers)


已找到 csv\all_etfs.csv，直接讀取資料。

--- 代碼清單 ---
['VOO', 'IVV', 'SPY', 'VTI', 'QQQ', 'VEA', 'VUG', 'IEFA', 'VTV', 'GLD', 'BND', 'IEMG', 'VXUS', 'AGG', 'IWF', 'VWO', 'VGT', 'IJH', 'VIG', 'IJR', 'VO', 'XLK', 'SCHD', 'SGOV', 'RSP', 'ITOT', 'BNDX', 'EFA', 'VYM', 'QQQM', 'IAU', 'IWM', 'VB', 'IWD', 'VT', 'VCIT', 'IVW', 'SCHX', 'VEU', 'SCHF', 'IBIT', 'IXUS', 'SCHG', 'BIL', 'IWR', 'XLF', 'SMH', 'IEF', 'QUAL', 'VV', 'IVE', 'JEPI', 'SPYG', 'BSV', 'IWB', 'DIA', 'MUB', 'VTEB', 'DFAC', 'VCSH', 'TLT', 'GOVT', 'VGIT', 'VONG', 'XLE', 'MBB', 'SCHB', 'DGRO', 'XLV', 'SPDW', 'JPST', 'SLV', 'IUSB', 'VNQ', 'JEPQ', 'VBR', 'SPYV', 'DYNF', 'LQD', 'GLDM', 'GDX', 'CGDV', 'VGK', 'XLI', 'EFV', 'ACWI', 'MGK', 'IDEV', 'VGSH', 'BIV', 'TQQQ', 'IUSG', 'EEM', 'VXF', 'JAAA', 'SHY', 'XLU', 'MDY', 'FBND', 'AVUV', 'XLC', 'SOXX', 'IUSV', 'USHY', 'FNDX', 'FNDF', 'USMV', 'DVY', 'MTUM', 'VOOG', 'AVEM', 'XLY', 'IGSB', 'VOE', 'SHV', 'VBK', 'RDVY', 'SCHA', 'SDY', 'CGGR', 'EMXC', 'EWJ', 'DFIV', 'IYW', 'IWP', 'VYMI', 'IEI', 'AVDV', 'DFUS'